# PLTR on UCI Credit Card Data

Dataset: UCI Machine Learning Repository, "Default of Credit Card Clients"

Variables:

- **ID**: ID of each client
- **LIMIT_BAL**: Amount of given credit in NT dollars (includes individual and family/supplementary credit
- **SEX**: Gender (1=male, 2=female)
- **EDUCATION**: (1=graduate school, 2=university, 3=high school, 4=others, 5=unknown, 6=unknown)
- **MARRIAGE**: Marital status (1=married, 2=single, 3=others)
- **AGE**: Age in years
- **PAY_0**: Repayment status in September, 2005 (-1=pay duly, 1=payment delay for one month, 2=payment delay for two months, … 8=payment delay for eight months, 9=payment delay for nine months and above)
- **PAY_2**: Repayment status in August, 2005 (scale same as above)
- **PAY_3**: Repayment status in July, 2005 (scale same as above)
- **PAY_4**: Repayment status in June, 2005 (scale same as above)
- **PAY_5**: Repayment status in May, 2005 (scale same as above)
- **PAY_6**: Repayment status in April, 2005 (scale same as above)
- **BILL_AMT1**: Amount of bill statement in September, 2005 (NT dollar)
- **BILL_AMT2**: Amount of bill statement in August, 2005 (NT dollar)
- **BILL_AMT3**: Amount of bill statement in July, 2005 (NT dollar)
- **BILL_AMT4**: Amount of bill statement in June, 2005 (NT dollar)
- **BILL_AMT5**: Amount of bill statement in May, 2005 (NT dollar)
- **BILL_AMT6**: Amount of bill statement in April, 2005 (NT dollar)
- **PAY_AMT1**: Amount of previous payment in September, 2005 (NT dollar)
- **PAY_AMT2**: Amount of previous payment in August, 2005 (NT dollar)
- **PAY_AMT3**: Amount of previous payment in July, 2005 (NT dollar)
- **PAY_AMT4**: Amount of previous payment in June, 2005 (NT dollar)
- **PAY_AMT5**: Amount of previous payment in May, 2005 (NT dollar)
- **PAY_AMT6**: Amount of previous payment in April, 2005 (NT dollar)
- **default.payment.next.month**: Default payment (1=yes, 0=no)

## Init

In [39]:
from pathlib import Path
import numpy as np
import pandas as pd

import itertools
import time

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss

from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from scipy import stats

In [2]:
FILE_PATH = "../data/UCI_Credit_Card.csv"

df = pd.read_csv(FILE_PATH)
df = df.rename(columns={"default.payment.next.month": "default"})

print("UCI Credit Card data:", df.shape)
display(df.head())


UCI Credit Card data: (30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


## EDA

The UCI text only defines -1 and 1–9. Public previews of the data show PAY_* values from -2 upwards, and a 0 as well. I believe -2 and 0 mean "no consumption" and "revolving, minimum paid", but that's community folklore, not documentation. Treat both as undefined and let the data speak.

In [3]:
print("Basic data check")
print("-"*30)
print(f"Missing:            {df.isna().sum().sum()}")
print(f"Duplicated rows:    {df.duplicated().sum()}")

display(df.iloc[:, :-1].describe().T[["min", "25%", "50%", "75%", "max"]])

for c in ["EDUCATION", "MARRIAGE", "PAY_0", "PAY_2"]:
    print(c, df[c].value_counts().sort_index().to_dict())

bills = [f"BILL_AMT{i}" for i in range(1,7)]
print("rows with negative bill:", (df[bills] < 0).any(axis=1).sum())

Basic data check
------------------------------
Missing:            0
Duplicated rows:    0


,min,25%,50%,75%,max
ID,1.0,7500.75,15000.5,22500.25,30000.0
LIMIT_BAL,10000.0,50000.00,140000.0,240000.00,1000000.0
SEX,1.0,1.00,2.0,2.00,2.0
EDUCATION,0.0,1.00,2.0,2.00,6.0
MARRIAGE,0.0,1.00,2.0,2.00,3.0
AGE,21.0,28.00,34.0,41.00,79.0
PAY_0,-2.0,-1.00,0.0,0.00,8.0
PAY_2,-2.0,-1.00,0.0,0.00,8.0
PAY_3,-2.0,-1.00,0.0,0.00,8.0
PAY_4,-2.0,-1.00,0.0,0.00,8.0


EDUCATION {0: 14, 1: 10585, 2: 14030, 3: 4917, 4: 123, 5: 280, 6: 51}
MARRIAGE {0: 54, 1: 13659, 2: 15964, 3: 323}
PAY_0 {-2: 2759, -1: 5686, 0: 14737, 1: 3688, 2: 2667, 3: 322, 4: 76, 5: 26, 6: 11, 7: 9, 8: 19}
PAY_2 {-2: 3782, -1: 6050, 0: 15730, 1: 28, 2: 3927, 3: 326, 4: 99, 5: 25, 6: 12, 7: 20, 8: 1}
rows with negative bill: 1930


- **Skew**: in the money columns, a mean far above the median means a heavy right tail. Trees don't care about that, but linear LR does.
- **Negative bills**: a negative statement likely means an overpayment or credit balance.
- **Duplicates**: report the number and don't drop anything blindly. Without an ID, identical rows can be genuinely different customers.

## Train-Test Split

In [4]:
train, test = train_test_split(df, test_size=.3, stratify=df["default"], random_state=42)
train, test = train.reset_index(drop=True), test.reset_index(drop=True)

print(f"Train:                  {train.shape}")
print(f"Test:                   {test.shape}")
print(f"Train default mean:     {train.default.mean().round(4)}")
print(f"Test default mean:      {test.default.mean().round(4)}")

assert train.default.mean().round(4) == test.default.mean().round(4)

Train:                  (21000, 25)
Test:                   (9000, 25)
Train default mean:     0.2212
Test default mean:      0.2212


In [5]:
# (a) default rate by latest repayment status
print("Default rate by latest repayment status")
print(train.groupby("PAY_0")["default"].agg(n="size", rate="mean").round(4))

# (b) Credit limit, in deciles
print("-"*30)
print("Credit limit, in deciles")
print(train.groupby(pd.qcut(train["LIMIT_BAL"], 10, duplicates="drop"), observed=True)["default"]
           .agg(n="size", rate="mean").round(3))

# (c) the odd category codes: do 0/5/6 behave like "4 = other"?
print("-"*30)
print("Do 0/5/6 behave like 4 = other?")
print(train.groupby("EDUCATION")["default"].agg(n="size", rate="mean").round(3))
print(train.groupby("MARRIAGE")["default"].agg(n="size", rate="mean").round(3))

# (d) A miniature of what PLTR hunts for: does the effect of lateness depend on the limit?
print("-"*30)
print("Does the effect of lateness depend on the limit?")
late = np.where(train["PAY_0"] >= 1, "PAY_0 >= 1", "PAY_0 <= 0")
lim  = np.where(train["LIMIT_BAL"] < train["LIMIT_BAL"].median(), "low limit", "high limit")
print(pd.crosstab(late, lim, values=train["default"], aggfunc="mean").round(3))
print(pd.crosstab(late, lim))

Default rate by latest repayment status
           n    rate
PAY_0               
-2      1933  0.1262
-1      3971  0.1697
 0     10285  0.1264
 1      2616  0.3421
 2      1887  0.6948
 3       213  0.7653
 4        51  0.6863
 5        13  0.3846
 6        10  0.6000
 7         8  0.7500
 8        13  0.4615
------------------------------
Credit limit, in deciles
                          n   rate
LIMIT_BAL                         
(9999.999, 30000.0]    2864  0.369
(30000.0, 50000.0]     2513  0.267
(50000.0, 70000.0]     1085  0.289
(70000.0, 100000.0]    2262  0.240
(100000.0, 140000.0]   1950  0.235
(140000.0, 180000.0]   2346  0.171
(180000.0, 210000.0]   1781  0.179
(210000.0, 270000.0]   2058  0.163
(270000.0, 360000.0]   2419  0.143
(360000.0, 1000000.0]  1722  0.116
------------------------------
Do 0/5/6 behave like 4 = other?
              n   rate
EDUCATION             
0            12  0.000
1          7379  0.195
2          9833  0.236
3          3440  0.251
4         

- (a) shows the shape of risk across PAY_0. If -2, -1 and 0 have similar low rates and risk jumps once delays begin, then coding PAY_0 as a plain number (where 0 → 1 is the same step as 1 → 2) fits poorly. That would be an argument for treating it as categories in the LR baseline, and it's evidence you should record.
- (b) shows whether risk falls steadily with the limit or has a cliff. This is the shape question from earlier: smooth slope or threshold.
- (c) is the evidence for or against merging. Only merge EDUCATION {0, 5, 6} into 4, and MARRIAGE 0 into 3, if the rates and counts look similar. Otherwise you'd be hiding signal.
- (d) is a 2×2 table of default rates. If the gap between "late" and "not late" is clearly different in the low-limit and high-limit columns, that's an interaction in miniature. If the gaps are about equal, this pair isn't a good PLTR candidate. I don't know which you'll see.

## Data cleaning

**PAY_0**:

- −2, −1, 0 are all "no delay" and sit at 12.6%, 17.0% and 12.6%. Level −1 is higher than 0 and −2. With these counts, that gap is about 6 standard errors, so it is real, though small next to what follows.
- Then the cliff: 34% at 1, 69% at 2, 77% at 3.
- Levels 4–8 are noise. They have 8–51 customers each, and rates like 38% (n=13) or 46% (n=13) have error bars of roughly ±13 points. Pool everything at 3 or above.

A straight line through PAY_0 would treat −2 → −1 as the same step as 1 → 2. It can't produce "flat, then a cliff, then a plateau". That is a threshold effect, which is exactly PLTR's territory. It also means a fair LR must encode PAY properly, otherwise PLTR's win would partly be a strawman. Recall the paper's own Taiwan LR scored worse than a constant guess on Brier.

**LIMIT_BAL**

It falls from 36.9% to 11.6% with no obvious cliff. The small bump in decile 3 is within noise. A log transform should capture it.

**EDUCATION and MARRIAGE**

- EDUCATION codes 4 and 5 match each other: 5.7% and 6.6%. Both differ sharply from the main groups, which sit at 19.5–25%.
- Codes 0 (n=12, zero defaults) and 6 (n=37) are too small to judge either way.
- A "merge only if they look similar" test therefore supports merging 5 into 4. It can't support keeping 0 and 6 separate.
- Pooling 0, 5 and 6 into 4 gives about 25 defaults in 348 rows (about 7%). That is a stable, coherent "other/unknown" group.
- Complete separation: a level with zero defaults would push an unpenalised LR coefficient toward minus infinity.
- The deeper issue is numeric coding. Codes 1, 2 and 3 have rising rates (19.5%, 23.6%, 25.1%), so a numeric slope would extrapolate to higher risk for codes 4–6, when the data shows about 7%. Treat EDUCATION as categories.
- MARRIAGE 0: the choice moves nothing. Keep it as its own level and treat it as noise-level evidence.

**The interaction table**

On the probability scale the late/not-late gap looks different by limit (39 points at low limit vs 29 at high). But LR works in log-odds, so compare on that scale:

|	|Low limit	|High limit	|Limit effect (log-odds)|
|-|-|-|-|
|Late (PAY_0 ≥ 1)	|56.5%	|40.4%	|0.65|
|Not late	|17.1%	|11.0%	|0.51|

The limit effect is 0.65 when late and 0.51 when not late. That difference of 0.14 has a standard error of about 0.08, so it is roughly 1.8 standard errors. That is weak evidence of an interaction, and the effects look nearly parallel. LR already produces probability gaps like these, because it adds in log-odds.

**The lesson**: judge interactions by comparing log-odds, not percentage points. It also adjusts my expectation. On this dataset, PLTR's gain will probably come more from thresholds (the PAY cliff) than from pair interactions. A well-encoded LR may close much of the gap.

In [6]:
def clean(d):
    d = d.copy()
    d["EDUCATION"] = d["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
    return d.drop(columns=["SEX"])

In [7]:
train_c, test_c = clean(train), clean(test)

X_train, y_train = train_c.drop(columns=["default", "ID"]), train_c["default"].values
X_test, y_test = test_c.drop(columns=["default", "ID"]), test_c["default"].values

cols_all = list(X_train.columns)
print(len(cols_all), "features:", cols_all)
assert len(cols_all) == 22 and "SEX" not in cols_all, "unexpected feature set"

print(f"X_train, y_train: {X_train.shape}, {y_train.shape}")
print(f"X_test, y_test:   {X_test.shape}, {y_test.shape}")

22 features: ['LIMIT_BAL', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
X_train, y_train: (21000, 22), (21000,)
X_test, y_test:   (9000, 22), (9000,)


## The two LR baselines

- **LR-A (naive)**: everything numeric, as-is, standardised. This is the careless version.
- **LR-B (careful)**: the encoding the evidence above justifies.

In [8]:
pay = ["PAY_0"] + [f"PAY_{i}" for i in range(2, 7)]
money = ["LIMIT_BAL"] + [f"BILL_AMT{i}" for i in range(1, 7)] + [f"PAY_AMT{i}" for i in range(1, 7)]
cats = ["EDUCATION", "MARRIAGE"]

def slog(a):
    # signed log: bills can be negative
    return np.sign(a) * np.log1p(np.abs(a))

def clip_pay(d):
    # rule-based: pool delays of 3+ months
    d = d.copy(); d[pay] = d[pay].clip(upper=3)
    return d

def lr_cv():
    return LogisticRegressionCV(Cs=np.logspace(-3, 2, 11), cv=5, scoring="roc_auc", max_iter=3000)

In [9]:
# LR-A: naive
lr_a = Pipeline([
    ("sc", StandardScaler()),
    ("lr", lr_cv())
])

# LR-B: careful
prep_b = ColumnTransformer([
    ("money", Pipeline([("log", FunctionTransformer(slog, feature_names_out="one-to-one")), ("sc", StandardScaler())]), money),
    ("age", StandardScaler(), ["AGE"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cats + pay)
])
lr_b = Pipeline([
    ("clip", FunctionTransformer(clip_pay)),
    ("prep", prep_b),
    ("lr", lr_cv())
])

- **Log on money**: these columns are heavily right-skewed, and the LIMIT_BAL curve looked smooth and log-like.
- **PAY clipped at 3, then one-hot**: this lets the model learn "flat, then cliff, then plateau", rather than forcing a line. The clip is a fixed rule, so it can't leak.
- **EDUCATION and MARRIAGE one-hot**: their codes aren't ordered by risk.
- **LogisticRegressionCV**: both models get the same tuned L2 penalty, which also handles the collinearity from one-hot dummies. The comparison stays fair.

## Choose the baseline by cross-validation on the training set

Decide which baseline PLTR must beat using training data only. If you picked the winner on the test set, you'd be selecting on it. A difference smaller than the ± is not a difference.

In [10]:
skf = StratifiedKFold(5, shuffle=True, random_state=0)
for name, m in [("LR-A naive", lr_a), ("LR-B careful", lr_b)]:
    s = cross_val_score(m, X_train, y_train, cv=skf, scoring="roc_auc", n_jobs=-1)
    print(f"{name:14s} CV AUC = {s.mean():.4f} ± {s.std():.4f}")

LR-A naive     CV AUC = 0.7246 ± 0.0167
LR-B careful   CV AUC = 0.7774 ± 0.0110


## Fit, then touch the test set once

In [11]:
def metrics(y, p):
    auc = roc_auc_score(y, p); fpr, tpr, _ = roc_curve(y, p)
    return {"AUC": auc, "Gini": 2*auc - 1, "KS": (tpr - fpr).max(),
            "Brier": brier_score_loss(y, p), "mean_pred": p.mean()}

def paired_boot(y, pa, pb, B=1000, seed=0):
    rng, diffs = np.random.default_rng(seed), []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y))
        diffs.append(roc_auc_score(y[i], pa[i]) - roc_auc_score(y[i], pb[i]))
    return np.mean(diffs), np.percentile(diffs, [2.5, 97.5])

In [12]:
lr_a.fit(X_train, y_train); lr_b.fit(X_train, y_train)

probs = {"LR-A": lr_a.predict_proba(X_test)[:, 1], "LR-B": lr_b.predict_proba(X_test)[:, 1]}
print(pd.DataFrame({k: metrics(y_test, p) for k, p in probs.items()}).T.round(4))
print("constant-prediction Brier:", round(y_test.mean() * (1 - y_test.mean()), 4), "| observed rate:", y_test.mean().round(4))
print("LR-B - LR-A AUC:", paired_boot(y_test, probs["LR-B"], probs["LR-A"]))

d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2129: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The d

         AUC    Gini      KS   Brier  mean_pred
LR-A  0.7135  0.4270  0.3724  0.1462     0.2214
LR-B  0.7644  0.5288  0.4180  0.1379     0.2202
constant-prediction Brier: 0.1723 | observed rate: 0.2212
LR-B - LR-A AUC: (np.float64(0.050826030681196496), array([0.04040858, 0.06173757]))


`metrics` and `paired_boot` are reused for PLTR.

Sanity-check LR-B's coefficients

In [13]:
names = lr_b.named_steps["prep"].get_feature_names_out()
coef  = pd.Series(lr_b.named_steps["lr"].coef_.ravel(), index=names)
print(coef.filter(like="PAY_0").round(3))                     # should echo the shape of table (a)
print(coef.reindex(coef.abs().sort_values(ascending=False).index).head(10).round(3))

cat__PAY_0_-2   -0.416
cat__PAY_0_-1   -0.297
cat__PAY_0_0    -0.940
cat__PAY_0_1    -0.109
cat__PAY_0_2     1.191
cat__PAY_0_3     0.553
dtype: float64
cat__PAY_0_2        1.191
cat__PAY_0_0       -0.940
cat__PAY_0_3        0.553
cat__PAY_0_-2      -0.416
cat__EDUCATION_4   -0.404
cat__PAY_0_-1      -0.297
cat__PAY_5_2        0.279
cat__PAY_3_2        0.240
cat__PAY_4_2        0.214
money__LIMIT_BAL   -0.202
dtype: float64


### Some notes on the result
- **LR-B beats LR-A solidly**. Cross-validated gap is .053 and the test gap is .051.
- **Test AUC sits about 0.011–0.013 below CV AUC for both models**. A single 9,000-row test set has an AUC standard error of roughly 0.007 (my estimate), so this is ordinary noise. It's not a leak, and since both models drop by the same amount, the comparison is unaffected.
- **Calibration is sound on average**. Mean predicted risk matches the observed rate (0.2203 vs 0.2212). Brier is 0.1379 against 0.1723 for a constant guess.
- **Versus the paper's Table 4**. Their linear LR scored 0.631 AUC. Your naive LR-A already scores 0.714, and LR-B scores 0.765 on test (0.777 in CV). Their PLTR scored 0.778 and their random forest 0.772. The protocols differ (5×2 CV on half the data), so don't over-compare. But it supports my suspicion that their LR baseline was weak. A careful LR already sits in PLTR's ballpark, so I expect little headroom.

## Step (a): Find out who earns the credit

If PLTR beats LR-A, is that the rules, or just the better encoding? To find out, change one thing at a time:

|Model	|What it is|
|-|-|
|LR-A	|Naive numeric LR (already fitted)|
|LR-B	|Careful-encoding LR, refitted with proper clipping|
|ALasso-B	|LR-B's design with PLTR's adaptive-lasso penalty, no rules|
|PLTR-A	|Paper-style: numeric features plus rules|
|PLTR-B	|LR-B's design plus rules|

Each contrast answers one question:

- **PLTR-A − LR-A**: the paper's claim, against a weak baseline.
- **PLTR-A − LR-B**: can PLTR automate what a careful analyst does by hand? This is the practically useful one.
- **ALasso-B − LR-B**: is any gain just from the penalty?
- **PLTR-B − ALasso-B**: do the rules add anything once encoding and penalty are held fixed? This is the cleanest test of the paper's idea.

### Clip outside the pipeline, refit LR-B

In [14]:
X_train_b, X_test_b = clip_pay(X_train), clip_pay(X_test)
prep_b = ColumnTransformer([
    ("money", Pipeline([("log", FunctionTransformer(slog, feature_names_out="one-to-one")), ("sc", StandardScaler())]), money),
    ("age", StandardScaler(), ["AGE"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cats + pay)
])
lr_b2 = Pipeline([
    ("prep", prep_b),
    ("lr", lr_cv())
]).fit(X_train_b, y_train)

print(lr_b2.named_steps["prep"].named_transformers_["cat"].categories_[2])

d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2129: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


[-2 -1  0  1  2  3]


### The PLTR code (replace any earlier versions)

- **Trees fit on training data only**: the rules use labels, so using test data would leak.
- **One leaf per pair**, redundant rules dropped: this follows the paper's method.
- **Ridge first, then L1**: the ridge coefficients become the adaptive-lasso weights (ν = 1), as the paper describes.
- `min_leaf_frac`: my guard against tiny leaves. Setting it to 1 row would be more faithful to the paper.
- `use_rules=False`: this gives you the ALasso-B control, with the same penalty and no rules.

In [15]:
from pltr import PLTR

### Fit

If it's too slow, set cv=5 and use `Cs=np.logspace(-2, 2, 6)`.

In [16]:
cols_all = list(X_train.columns)
models = {
    "LR-A":     lr_a,     # already fitted on raw Xtr
    "LR-B":     lr_b2,
    "ALasso-B": PLTR(prep_b, cols_all, use_rules=False).fit(X_train_b, y_train),
    "PLTR-A":   PLTR(StandardScaler(), cols_all).fit(X_train, y_train),
    "PLTR-B":   PLTR(prep_b, cols_all).fit(X_train_b, y_train),
}

d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2102: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' and 'Cs' instead. Use l1_ratios=(0,) instead of penalty='l2', l1_ratios=(1,) instead of penalty='l1', l1_ratios set to floats between 0 and 1 instead of penalty='elasticnet', and Cs=(np.inf,) instead of penalty=None.
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2129: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove

### Evaluate on test set

In [17]:
# the A models use raw columns, the B models clipped ones
Xmap = {"LR-A": X_test, "PLTR-A": X_test}
probs = {k: m.predict_proba(Xmap.get(k, X_test_b))[:, 1] for k, m in models.items()}
print(pd.DataFrame({k: metrics(y_test, p) for k, p in probs.items()}).T.round(4))

for a, b in [("PLTR-A","LR-A"), ("PLTR-A","LR-B"), ("ALasso-B","LR-B"), ("PLTR-B","ALasso-B"), ("PLTR-B","LR-B")]:
    m, ci = paired_boot(y_test, probs[a], probs[b])
    print(f"{a} - {b}: dAUC = {m:+.4f}   95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}]")

for name in ["PLTR-A", "PLTR-B"]:
    print("\n==", name)
    models[name].explain(models[name].base_.get_feature_names_out())

             AUC    Gini      KS   Brier  mean_pred
LR-A      0.7135  0.4270  0.3724  0.1462     0.2214
LR-B      0.7644  0.5288  0.4180  0.1379     0.2202
ALasso-B  0.7645  0.5289  0.4154  0.1377     0.2204
PLTR-A    0.7755  0.5511  0.4320  0.1362     0.2208
PLTR-B    0.7732  0.5465  0.4290  0.1363     0.2205
PLTR-A - LR-A: dAUC = +0.0620   95% CI [+0.0512, +0.0732]
PLTR-A - LR-B: dAUC = +0.0112   95% CI [+0.0062, +0.0163]
ALasso-B - LR-B: dAUC = +0.0001   95% CI [-0.0010, +0.0010]
PLTR-B - ALasso-B: dAUC = +0.0088   95% CI [+0.0052, +0.0126]
PLTR-B - LR-B: dAUC = +0.0089   95% CI [+0.0053, +0.0124]

== PLTR-A
non-zero: 42 of 168 | rules kept: 34 (pair trees: 26, single-variable: 8)
-1.173  RULE: PAY_0 <= 1.50 AND PAY_2 <= 1.50
-1.105  RULE: PAY_0 <= 1.50
-0.568  RULE: PAY_4 <= 0.50 AND PAY_6 <= 1.00
+0.440  RULE: PAY_4 <= 0.50 AND EDUCATION <= 3.50
-0.392  RULE: PAY_2 <= 1.50 AND PAY_4 <= 0.50
-0.352  RULE: PAY_5 <= 1.00
+0.342  RULE: LIMIT_BAL <= 145000.00
+0.329  RULE: PAY_2 <= 1.5

**What to look at**
1. **The metrics table**. Check Brier and `mean_pred` first. The lasso shrinks coefficients, which can slightly distort calibration.
2. **Each paired difference against its own question**. If a 95% interval contains 0, you have no evidence of a difference.
3. **Which rules survive**. How many were kept, and how many are pair trees versus single-variable rules? If PLTR-A's top rules are PAY thresholds, it has rediscovered what LR-B's dummies encoded by hand. Do any pair rules make credit sense?
4. **Whether the lasso pruned much**. In my simulation it kept 15 of 20 terms, so don't expect aggressive sparsity.
5. **Limits of this test**. It's one split of 9,000 rows, whereas the paper used 5×2 cross-validation. If a difference is small, a later step is repeated splits.

### What the ablation shows

- PLTR-A − LR-B: +0.0112 (CI [+0.0062, +0.0163]). Singles-only PLTR-A was 0.7684, so the genuine interactions add roughly +0.007 AUC. That's modest but plausible, and I'd call the effect real if it survives 5×2 CV.
- The penalty alone still adds nothing: ALasso-B − LR-B = −0.0003, CI spanning zero. Rules are the whole story.
- PLTR-A is sparser and cleaner: 41 of 168 non-zero, against 85 of 187 before. That's closer to something you could defend as interpretable.
- PLTR-A vs PLTR-B (0.7755 vs 0.7732): no CI yet, so don't interpret it. The one-hot PAY columns in B probably duplicate what the PAY rules encode, but that is an untested hypothesis.
- Calibration is fine: mean_pred is 0.2208 against a base rate of 0.221.

**What the rules mean in credit terms**

- The odd bottom-left cell from before has gone. PAY_0 <= 1.5 AND PAY_2 <= 1.5 (−1.173) now sits beside PAY_0 <= 1.5 (−1.105), so "not late on either" is the lowest-risk cell, which is sensible.
- Some signs are still counter-intuitive, such as PAY_2 <= 1.5 AND BILL_AMT1 <= 783 at +0.329 and PAY_4 <= 0.5 AND EDUCATION <= 3.5 at +0.439. The latter is probably the complement of the tiny EDUCATION_4 group combined with PAY_4, and the former may be low-balance, low-activity accounts. Treat both as hypotheses. The coefficients are conditional on which overlapping rules survived, so read risk across a grid, not coefficient by coefficient.
- Your test set is now spent. You've looked at it several times, with a bug fix and two refits. From here on, modelling decisions must come from cross-validation.


## Step (b): 5x2 CV

5 repetitions × a different random 50/50 split each time × train on one half, test on the other, then swap. That gives 10 AUCs per model, paired by fold.

But a "Naïve t-test" is Over-Confident: When comparing Model A and Model B across standard 10-fold cross-validation, a regular (naïve) t-test assumes all 10 score differences are completely independent.
- Because 80% of the training data overlaps between folds, the scores look artificially consistent.
- The test underestimates the true variation and produces a tiny p-value, causing you to falsely conclude that one model is better than the other when the difference is just random noise.

**Alpaydin's F Statistic**

$$
F = \frac{\sum_{i=1}^5 \sum_{j=1}^2 p_{ij}^2}{2\sum_{i=1}^5 s_i^2}
$$

- **$p_{ij}$ (The Performance Difference)**: On repetition i (from 1 to 5) and swapped sitting j (1 or 2), you subtract Model B's AUC from Model A's AUC. You end up with 10 difference scores.
- **$\={p}_{i}$ (The Repetition Average)**: For a single 50/50 split i, this is simply the average of the two swapped difference scores: $(p_{i1} + p_{i2})/2$.
- **$s_i^2$ (Within-Repetition Variance)**: This measures how much the performance gap swings when you swap the training and test halves. If Model A beats Model B by +0.05 on Half A, and beats it by +0.05 on Half B, the variance $s_i^2$ is 0 (rock-solid consistency). If it swings wildly between +0.10 and -0.02, $$s_i^2$ is large.
- **Numerator**: The **Signal**. It adds up all the squared AUC differences across all 10 runs. A bigger gap between the two models makes the top number larger.
- **Denominator**: The **Noise**. It adds up the instability (variance) across the 5 splits. If the models' rankings flip-flop whenever you swap the halves, the bottom number gets large and shrinks the F score.

**(10, 5) Degrees of Freedom**

- **10**: the numerator, there are 5 * 2 = 10 $p_{ij}$ and none of them are locked by a shared average.
- **5**: the denominator, 5 repetitions, each has 2 swapped scores ($p_{i1}$ and $p_{i2}$) compared against their own average ($\={p}_i$), so if $p_{i1}$ is chosen freely, the second $p$ is locked by $\={p}_i$, hence the degree of freedom is 1. 5 repetitions -> $5 * 1 = 5$ degrees of freedom.

To turn that $F$ number into a final p-value (to decide if the winner is statistically real), you look up your $F$ score on an **F-distribution table** using 10 (representing the 10 squared differences in the numerator) and 5 (representing the 5 variance estimates in the denominator).

If your calculated $F$ value exceeds the critical threshold for (10, 5) degrees of freedom—which is roughly 4.74 for 95% confidence—you can safely declare that one model genuinely outperforms the other.

In [18]:
from pltr import RuleExtractor

In [22]:
# --- guard: the fixed extractor must be active
_probe = RuleExtractor(0.01).fit(X_train[cols_all].values.astype(float)[:3000], y_train[:3000])
assert "pair_one_variable" in _probe.stats_, "old RuleExtractor still active - restart and re-run definitions"

_Full = RuleExtractor
class SinglesOnly(_Full):
    def fit(self, X, y):
        super().fit(X, y)
        self.rules_ = [r for r in self.rules_ if len(r[0]) == 1]
        return self

rc = models["PLTR-A"].rule_cols
Cs_fast = np.logspace(-2, 2, 6)

def mk(base, use_rules):
    return PLTR(clone(base), rc, use_rules=use_rules, Cs=Cs_fast, cv=5)

In [26]:
specs = {   # name: (builder, extractor class used while fitting)
    "LR-A":        (lambda: clone(models["LR-A"]), _Full),
    "LR-B":        (lambda: clone(models["LR-B"]), _Full),
    "ALasso-B":    (lambda: mk(models["ALasso-B"].base, False), _Full),
    "PLTR-A-sing": (lambda: mk(models["PLTR-A"].base, True), SinglesOnly),
    "PLTR-A":      (lambda: mk(models["PLTR-A"].base, True), _Full),
    "PLTR-B":      (lambda: mk(models["PLTR-B"].base, True), _Full),
}

Xall = pd.concat([X_train, X_test]).reset_index(drop=True)
yall = pd.Series(np.concatenate([y_train, y_test]))

if "cv_res" not in globals(): cv_res = {}          # resumable: (rep, fold, model) -> (auc, brier)
t0 = time.time()
for rep in range(5):
    skf = StratifiedKFold(2, shuffle=True, random_state=rep)
    for fold, (i_train, i_test) in enumerate(skf.split(Xall, yall)):
        for name, (build, ext) in specs.items():
            if (rep, fold, name) in cv_res: continue
            globals()["RuleExtractor"] = ext
            try:
                m = build().fit(Xall.iloc[i_train], yall.iloc[i_train])
            finally:
                globals()["RuleExtractor"] = _Full
            p = m.predict_proba(Xall.iloc[i_test])[:, 1]
            cv_res[(rep, fold, name)] = (roc_auc_score(yall.iloc[i_test], p), brier_score_loss(yall.iloc[i_test], p))
        print(f"rep {rep} fold {fold} done  ({(time.time()-t0)/60:.1f} min)", flush=True)
        pd.Series(cv_res).to_pickle("cv_res.pkl")   # checkpoint

d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2129: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The d

rep 0 fold 0 done  (1.5 min)


d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2129: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The d

rep 0 fold 1 done  (2.5 min)


d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2129: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The d

rep 1 fold 0 done  (3.5 min)


... (10 outputs left out)

d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2129: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The d

rep 4 fold 0 done  (10.6 min)


d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2129: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The d

rep 4 fold 1 done  (12.3 min)


In [29]:
# ---- results
auc = pd.DataFrame({n: [cv_res[(r, f, n)][0] for r in range(5) for f in range(2)] for n in specs})
print(auc.agg(["mean", "std"]).round(4).T)

def f_test(a, b):
    d = (auc[a] - auc[b]).values.reshape(5, 2)
    s2 = ((d - d.mean(1, keepdims=True)) ** 2).sum(1)
    F = (d ** 2).sum() / (2 * s2.sum())
    return d.mean(), (d > 0).sum(), stats.f.sf(F, 10, 5)

print("\npaired differences (A - B):  mean dAUC | folds A>B of 10 | Alpaydin F p-value")
for a, b in [("PLTR-A","LR-B"), ("PLTR-A","LR-A"), ("PLTR-A","PLTR-A-sing"), ("PLTR-A","ALasso-B"),
             ("PLTR-A-sing","ALasso-B"), ("PLTR-B","PLTR-A"), ("ALasso-B","LR-B")]:
    m_, w, p_ = f_test(a, b)
    print(f"{a:>12} - {b:<12} {m_:+.4f} | {w:>2}/10 | p = {p_:.3f}")

               mean     std
LR-A         0.7216  0.0052
LR-B         0.7738  0.0047
ALasso-B     0.7735  0.0045
PLTR-A-sing  0.7759  0.0035
PLTR-A       0.7759  0.0035
PLTR-B       0.7752  0.0040

paired differences (A - B):  mean dAUC | folds A>B of 10 | Alpaydin F p-value
      PLTR-A - LR-B         +0.0021 |  8/10 | p = 0.328
      PLTR-A - LR-A         +0.0543 | 10/10 | p = 0.000
      PLTR-A - PLTR-A-sing  +0.0000 |  7/10 | p = 0.637
      PLTR-A - ALasso-B     +0.0024 |  8/10 | p = 0.208
 PLTR-A-sing - ALasso-B     +0.0024 |  8/10 | p = 0.208
      PLTR-B - PLTR-A       -0.0007 |  3/10 | p = 0.572
    ALasso-B - LR-B         -0.0003 |  1/10 | p = 0.547


- Under 5×2 CV, PLTR-A beats LR-B by only +0.0021 AUC (8/10 folds, p = 0.33). That is not distinguishable from zero. The single-split +0.0112 was mostly an easy test set for PLTR: LR-B scored 0.7644 there but 0.7738 on average across the ten CV folds.
- Encoding is the big effect. PLTR-A − LR-A is +0.054 in 10/10 folds, and LR-B closes almost all of that gap.
- The penalty alone adds nothing: ALasso-B − LR-B = −0.0003, with LR-B ahead in 9 of 10 folds.
- Rules add a small, unproven amount: about +0.002 over LR-B or ALasso-B, winning 8/10 folds. The direction is consistent, but the F-test has 5 denominator degrees of freedom and little power. I'd write "suggestive, not established".
- PLTR-A vs PLTR-B is a wash (−0.0007, p = 0.57), so my one-hot duplication hypothesis has no support.
- PLTR is more stable across folds: its std is 0.0035 against 0.0047 for LR-B. That is a modest point in its favour, not a significance test.

### Checking

In [31]:
_Full_ = _Full
fits = {}
for name, ext in [("all", _Full), ("sing", SinglesOnly)]:
    globals()["RuleExtractor"] = ext
    try:
        fits[name] = mk(models["PLTR-A"].base, True).fit(X_train, y_train)
    finally:
        globals()["RuleExtractor"] = _Full_
for name, m in fits.items():
    coef, nb = m.lasso_.coef_.ravel(), m.nb_
    n_pair_all = sum(len(c) == 2 for c, _, _ in m.rules_.rules_)
    n_pair_kept = sum(len(c) == 2 and coef[nb + k] != 0 for k, (c, _, _) in enumerate(m.rules_.rules_))
    print(f"{name:>5}: columns {len(coef)} | pair rules available {n_pair_all} | pair rules kept {n_pair_kept} "
          f"| best C {m.lasso_.C_[0]:.3g}")
pa, ps = (fits[k].predict_proba(X_test)[:, 1] for k in ("all", "sing"))
print("max |p_all - p_sing|:", np.abs(pa - ps).max().round(4), "| corr:", np.corrcoef(pa, ps)[0, 1].round(5))

d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2102: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' and 'Cs' instead. Use l1_ratios=(0,) instead of penalty='l2', l1_ratios=(1,) instead of penalty='l1', l1_ratios set to floats between 0 and 1 instead of penalty='elasticnet', and Cs=(np.inf,) instead of penalty=None.
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2129: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove

  all: columns 168 | pair rules available 124 | pair rules kept 31 | best C 0.398
 sing: columns 168 | pair rules available 124 | pair rules kept 31 | best C 0.398
max |p_all - p_sing|: 0.0007 | corr: 1.0


### Explicit extractor, seeded solver, rerun the key comparison

In [35]:
class PLTRx(PLTR):
    """PLTR with an explicit extractor class and a seeded lasso solver."""
    def __init__(self, base, rule_cols, use_rules=True, min_leaf_frac=.01,
                 Cs=np.logspace(-2, 2, 9), cv=10, extractor=None):
        super().__init__(base, rule_cols, use_rules, min_leaf_frac, Cs, cv)
        self.extractor = extractor

    def fit(self, X, y):
        self.base_ = clone(self.base).fit(X, y)
        self.nb_ = self._dense(self.base_.transform(X)).shape[1]
        if self.use_rules:
            self.rules_ = self.extractor(self.min_leaf_frac).fit(
                X[self.rule_cols].values.astype(float), y)
        D = self._design(X)
        ridge = LogisticRegression(C=1.0, max_iter=3000).fit(D, y)
        self.w_ = np.abs(ridge.coef_.ravel()) + 1e-6
        self.lasso_ = LogisticRegressionCV(
            Cs=list(self.Cs), cv=StratifiedKFold(self.cv, shuffle=True, random_state=0),
            penalty="l1", solver="liblinear", scoring="roc_auc",
            max_iter=500, n_jobs=-1, random_state=0).fit(D * self.w_, y)
        return self

rc, base_A = models["PLTR-A"].rule_cols, models["PLTR-A"].base
Cs_fast = np.logspace(-2, 2, 6)
mk2 = lambda ext: PLTRx(clone(base_A), rc, True, Cs=Cs_fast, cv=5, extractor=ext)

# 1) Sanity check: the two variants must now genuinely differ
chk = {n: mk2(e).fit(X_train, y_train) for n, e in [("all", _Full), ("sing", SinglesOnly)]}
for n, m in chk.items():
    print(n, "columns:", m.nb_ + len(m.rules_.rules_),
          "| pair rules:", sum(len(c) == 2 for c, _, _ in m.rules_.rules_))
assert chk["all"].nb_ + len(chk["all"].rules_.rules_) > chk["sing"].nb_ + len(chk["sing"].rules_.rules_), "swap still failing"

# 2) 5x2 CV, same folds as before, new keys so old results stay untouched
t0 = time.time()
for rep in range(5):
    skf = StratifiedKFold(2, shuffle=True, random_state=rep)
    for fold, (i_tr, i_te) in enumerate(skf.split(Xall, yall)):
        for name, ext in [("P-sing", SinglesOnly), ("P-all", _Full)]:
            if (rep, fold, name) in cv_res: continue
            p = mk2(ext).fit(Xall.iloc[i_tr], yall.iloc[i_tr]).predict_proba(Xall.iloc[i_te])[:, 1]
            cv_res[(rep, fold, name)] = (roc_auc_score(yall.iloc[i_te], p),
                                         brier_score_loss(yall.iloc[i_te], p))
        print(f"rep {rep} fold {fold} ({(time.time()-t0)/60:.1f} min)", flush=True)
        pd.Series(cv_res).to_pickle("cv_res.pkl")

names = ["LR-B", "ALasso-B", "P-sing", "P-all"]
auc = pd.DataFrame({n: [cv_res[(r, f, n)][0] for r in range(5) for f in range(2)] for n in names})
print(auc.agg(["mean", "std"]).round(4).T)
for a, b in [("P-all", "P-sing"), ("P-sing", "LR-B"), ("P-all", "LR-B"), ("P-all", "ALasso-B")]:
    m_, w, p_ = f_test(a, b)
    print(f"{a:>7} - {b:<9} {m_:+.4f} | {w:>2}/10 | p = {p_:.3f}")

d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2102: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' and 'Cs' instead. Use l1_ratios=(0,) instead of penalty='l2', l1_ratios=(1,) instead of penalty='l1', l1_ratios set to floats between 0 and 1 instead of penalty='elasticnet', and Cs=(np.inf,) instead of penalty=None.
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2129: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove

all columns: 168 | pair rules: 124
sing columns: 44 | pair rules: 0


d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2102: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' and 'Cs' instead. Use l1_ratios=(0,) instead of penalty='l2', l1_ratios=(1,) instead of penalty='l1', l1_ratios set to floats between 0 and 1 instead of penalty='elasticnet', and Cs=(np.inf,) instead of penalty=None.
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2129: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove

rep 0 fold 0 (0.5 min)


d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2102: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' and 'Cs' instead. Use l1_ratios=(0,) instead of penalty='l2', l1_ratios=(1,) instead of penalty='l1', l1_ratios set to floats between 0 and 1 instead of penalty='elasticnet', and Cs=(np.inf,) instead of penalty=None.
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2129: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove

rep 0 fold 1 (0.9 min)


... (12 outputs left out)

d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2102: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' and 'Cs' instead. Use l1_ratios=(0,) instead of penalty='l2', l1_ratios=(1,) instead of penalty='l1', l1_ratios set to floats between 0 and 1 instead of penalty='elasticnet', and Cs=(np.inf,) instead of penalty=None.
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2129: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove

rep 4 fold 0 (4.1 min)


d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2102: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' and 'Cs' instead. Use l1_ratios=(0,) instead of penalty='l2', l1_ratios=(1,) instead of penalty='l1', l1_ratios set to floats between 0 and 1 instead of penalty='elasticnet', and Cs=(np.inf,) instead of penalty=None.
  warnings.warn(
d:\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2129: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove

rep 4 fold 1 (4.8 min)
            mean     std
LR-B      0.7738  0.0047
ALasso-B  0.7735  0.0045
P-sing    0.7710  0.0034
P-all     0.7759  0.0035
  P-all - P-sing    +0.0050 | 10/10 | p = 0.041
 P-sing - LR-B      -0.0028 |  1/10 | p = 0.441
  P-all - LR-B      +0.0021 |  8/10 | p = 0.329
  P-all - ALasso-B  +0.0024 |  8/10 | p = 0.208


Most of PLTR's advantage over naive logistic regression (+0.054 AUC) is recovered by encoding PAY sensibly. Over a well-encoded logistic regression, full PLTR adds about +0.002 AUC (8/10 folds, not significant). Pair rules add about +0.005 over singles-only PLTR (10/10 folds, p = 0.04), but singles-only PLTR is itself slightly below the well-encoded baseline.

## Step (c): WoE scorecard

In [36]:
from woe import WoE
from sklearn.pipeline import make_pipeline

In [37]:
mk_woe = lambda: make_pipeline(WoE(), LogisticRegression(C=1.0, max_iter=2000))
Xc = Xall[cols_all]

for rep in range(5):
    skf = StratifiedKFold(2, shuffle=True, random_state=rep)
    for fold, (i_tr, i_te) in enumerate(skf.split(Xall, yall)):
        if (rep, fold, "WoE-LR") in cv_res: continue
        m = mk_woe().fit(Xc.iloc[i_tr], yall.iloc[i_tr])
        p = m.predict_proba(Xc.iloc[i_te])[:, 1]
        cv_res[(rep, fold, "WoE-LR")] = (roc_auc_score(yall.iloc[i_te], p), brier_score_loss(yall.iloc[i_te], p))
pd.Series(cv_res).to_pickle("cv_res.pkl")

names = list(specs) + ["WoE-LR"]
auc = pd.DataFrame({n: [cv_res[(r, f, n)][0] for r in range(5) for f in range(2)] for n in names})
print(auc.agg(["mean", "std"]).round(4).T)
for a, b in [("WoE-LR", "LR-B"), ("PLTR-A", "WoE-LR"), ("WoE-LR", "LR-A")]:
    m_, w, p_ = f_test(a, b)
    print(f"{a:>8} - {b:<8} {m_:+.4f} | {w:>2}/10 | p = {p_:.3f}")

               mean     std
LR-A         0.7216  0.0052
LR-B         0.7738  0.0047
ALasso-B     0.7735  0.0045
PLTR-A-sing  0.7759  0.0035
PLTR-A       0.7759  0.0035
PLTR-B       0.7752  0.0040
WoE-LR       0.7697  0.0044
  WoE-LR - LR-B     -0.0041 |  0/10 | p = 0.046
  PLTR-A - WoE-LR   +0.0062 | 10/10 | p = 0.010
  WoE-LR - LR-A     +0.0481 | 10/10 | p = 0.000


Ignore the old PLTR-A-sing row, which duplicated PLTR-A. In the earlier WoE cell, compare against P-all, not PLTR-A:

In [38]:
if "WoE-LR" not in {k[2] for k in cv_res}:
    print("Run the WoE cell from my earlier message first (it defines WoE and mk_woe).")
else:
    names = ["LR-B", "P-sing", "P-all", "WoE-LR"]
    auc = pd.DataFrame({n: [cv_res[(r, f, n)][0] for r in range(5) for f in range(2)] for n in names})
    print(auc.agg(["mean", "std"]).round(4).T)
    for a, b in [("WoE-LR", "LR-B"), ("P-all", "WoE-LR"), ("WoE-LR", "P-sing")]:
        m_, w, p_ = f_test(a, b)
        print(f"{a:>7} - {b:<7} {m_:+.4f} | {w:>2}/10 | p = {p_:.3f}")

          mean     std
LR-B    0.7738  0.0047
P-sing  0.7710  0.0034
P-all   0.7759  0.0035
WoE-LR  0.7697  0.0044
 WoE-LR - LR-B    -0.0041 |  0/10 | p = 0.046
  P-all - WoE-LR  +0.0062 | 10/10 | p = 0.010
 WoE-LR - P-sing  -0.0013 |  4/10 | p = 0.496


The only gaps that are consistent across folds and bigger than solver noise are P-all over P-sing (+0.005), P-all over WoE-LR (+0.006) and LR-B over WoE-LR (+0.004). P-all over LR-B (+0.002) remains unproven. Applying Bonferroni for 4 comparisons (threshold 0.0125), only the P-all − WoE-LR result (p = 0.010) passes, and only just. P-all − P-sing (p = 0.041) fails. I'd still report it as "consistent in 10/10 folds, not significant after correction".

## Step (d): how much non-additive signal exists?

LR-B is a ruler measuring each factor separately and adding the results. A boosted tree ensemble is a team of inspectors who can check any combination of factors. If the team does not clearly beat the ruler, there is little interaction signal left for PLTR's pair rules to find.

A random forest averages many deep, independent trees. Gradient boosting adds small trees one after another, each correcting the previous ones' errors. Both capture interactions automatically.

Both are fitted on the raw 22 columns with no encoding, since trees don't care about scale. I'm using sklearn's `HistGradientBoostingClassifier` because it is fast on CPU, with early stopping inside each training half. Neither model is tuned, so treat the results as a reference point, not a ceiling.

In [40]:
Xt = Xall[cols_all]
refs = {
    "RF":  lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=0),
    "HGB": lambda: HistGradientBoostingClassifier(learning_rate=0.05, max_iter=400, max_leaf_nodes=15, l2_regularization=1.0, early_stopping=True, validation_fraction=0.15, n_iter_no_change=20, random_state=0),
}
t0 = time.time()
for rep in range(5):
    skf = StratifiedKFold(2, shuffle=True, random_state=rep)
    for fold, (i_tr, i_te) in enumerate(skf.split(Xall, yall)):
        for name, build in refs.items():
            if (rep, fold, name) in cv_res: continue
            p = build().fit(Xt.iloc[i_tr], yall.iloc[i_tr]).predict_proba(Xt.iloc[i_te])[:, 1]
            cv_res[(rep, fold, name)] = (roc_auc_score(yall.iloc[i_te], p), brier_score_loss(yall.iloc[i_te], p))
        print(f"rep {rep} fold {fold} ({(time.time()-t0)/60:.1f} min)", flush=True)
pd.Series(cv_res).to_pickle("cv_res.pkl")

names = ["LR-B", "P-all", "RF", "HGB"]
auc = pd.DataFrame({n: [cv_res[(r, f, n)][0] for r in range(5) for f in range(2)] for n in names})
print(auc.agg(["mean", "std"]).round(4).T)
for a, b in [("HGB", "LR-B"), ("HGB", "P-all"), ("RF", "P-all"), ("HGB", "RF")]:
    m_, w, p_ = f_test(a, b)
    print(f"{a:>4} - {b:<6} {m_:+.4f} | {w:>2}/10 | p = {p_:.3f}")

rep 0 fold 0 (0.0 min)
rep 0 fold 1 (0.1 min)
rep 1 fold 0 (0.1 min)
rep 1 fold 1 (0.1 min)
rep 2 fold 0 (0.2 min)
rep 2 fold 1 (0.2 min)
rep 3 fold 0 (0.3 min)
rep 3 fold 1 (0.3 min)
rep 4 fold 0 (0.3 min)
rep 4 fold 1 (0.4 min)
         mean     std
LR-B   0.7738  0.0047
P-all  0.7759  0.0035
RF     0.7816  0.0038
HGB    0.7801  0.0035
 HGB - LR-B   +0.0064 | 10/10 | p = 0.019
 HGB - P-all  +0.0042 | 10/10 | p = 0.000
  RF - P-all  +0.0057 | 10/10 | p = 0.001
 HGB - RF     -0.0014 |  2/10 | p = 0.324


The headroom is itself small. RF − LR-B is about +0.008 (0.7816 vs 0.7738). The Taiwan data is noisy, and nothing in your runs gets past roughly 0.78 AUC, so the whole non-additive prize is under a hundredth of AUC. Against that:

- P-all captures +0.0021 of the +0.0064 that HGB gains over LR-B, about a third.
- HGB − P-all is +0.0042 in 10/10 folds. This gap is real, and it is bigger than PLTR's own gain over LR-B.
- The p-values show why the p-value is not the effect size. HGB − LR-B is larger (+0.0064) but has p = 0.019, while HGB − P-all is smaller but has p ≈ 0.000, because its fold differences are more consistent. Read the mean and the win count first and the p-value last.

What this does to the claims

- "Interactions exist here": weakly supported. Trees beat every linear-family model, and RF ≈ HGB (−0.0014, 2/10, so no real difference) suggests the ceiling is near 0.78.
- "PLTR's pair rules capture them": only partly. They recover about a third of the tree advantage, and the PLTR-A − P-sing gap was +0.005.
- The confound that blocks a firm conclusion: RF and HGB gain from non-additivity and also from flexible nonlinear shapes, such as the money columns and the PAY categories. A tree's win over LR-B does not prove interactions. We need an additive but flexible model as a control.

### separate non-additivity from nonlinearity

Give the team of inspectors a rule that each one may look at only one factor. They can still draw any curve per factor, but they cannot combine factors. If they match the full team, the extra signal was curvature, not interaction.

`interaction_cst` tells HGB which features may appear together in a tree. One feature per group gives an additive model, a boosted GAM. max_depth=2 allows at most pairwise combinations within each tree, the closest tree analogue of PLTR's pair rules.

In [41]:
d = len(cols_all)
common = dict(learning_rate=0.05, l2_regularization=1.0, early_stopping=True, validation_fraction=0.15, n_iter_no_change=30, random_state=0)
refs2 = {
    "HGB-add": lambda: HistGradientBoostingClassifier(max_iter=1500, max_leaf_nodes=4, interaction_cst=[[j] for j in range(d)], **common),
    "HGB-d2":  lambda: HistGradientBoostingClassifier(max_iter=800, max_depth=2, max_leaf_nodes=4, **common),
}
for rep in range(5):
    skf = StratifiedKFold(2, shuffle=True, random_state=rep)
    for fold, (i_tr, i_te) in enumerate(skf.split(Xall, yall)):
        for name, build in refs2.items():
            if (rep, fold, name) in cv_res: continue
            p = build().fit(Xt.iloc[i_tr], yall.iloc[i_tr]).predict_proba(Xt.iloc[i_te])[:, 1]
            cv_res[(rep, fold, name)] = (roc_auc_score(yall.iloc[i_te], p), brier_score_loss(yall.iloc[i_te], p))
pd.Series(cv_res).to_pickle("cv_res.pkl")

names = ["LR-B", "P-all", "HGB-add", "HGB-d2", "HGB"]
auc = pd.DataFrame({n: [cv_res[(r, f, n)][0] for r in range(5) for f in range(2)] for n in names})
print(auc.agg(["mean", "std"]).round(4).T)
for a, b in [("HGB-add", "LR-B"), ("HGB", "HGB-add"), ("HGB-d2", "HGB-add"), ("P-all", "HGB-add")]:
    m_, w, p_ = f_test(a, b)
    print(f"{a:>8} - {b:<8} {m_:+.4f} | {w:>2}/10 | p = {p_:.3f}")

           mean     std
LR-B     0.7738  0.0047
P-all    0.7759  0.0035
HGB-add  0.7729  0.0035
HGB-d2   0.7784  0.0037
HGB      0.7801  0.0035
 HGB-add - LR-B     -0.0009 |  4/10 | p = 0.704
     HGB - HGB-add  +0.0073 | 10/10 | p = 0.002
  HGB-d2 - HGB-add  +0.0055 | 10/10 | p = 0.003
   P-all - HGB-add  +0.0030 |  9/10 | p = 0.140


## Write-up skeleton

My findings so far:

1. Naive LR vs PLTR gap (+0.054, 10/10) is mostly encoding. LR-B recovers almost all of it.
2. The adaptive-lasso penalty alone adds nothing (−0.0003).
3. Full PLTR vs LR-B: +0.0021, not significant. Pair rules over singles-only: +0.005, 10/10, p = 0.04 (uncorrected).
4. A simple WoE scorecard is 0.004 below LR-B. Binning loses information.
5. Control experiment: a flexible additive model ≈ LR-B, so curvature is not the issue. Depth-2 boosting beats it by +0.0055, so pairwise interactions are real.

Limits to state: one dataset, one noisy target with a ceiling near 0.78, untuned baselines, a paper protocol I only approximated, and a test set that was used repeatedly before the CV.